In [29]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')
class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"너는 사용자의 질문에 친철하고 정확하게 답변하는 시스템 입니다., Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,            
        )
        return response.choices[0].message.content
llm = OpenAILLM()    

In [47]:
import json
from  textwrap import dedent  #  들여쓰기 indent를 제거
query = '어제 삼성전자 주식종가 를 조회하고 해당 종가와 비슷한종목을 뉴스에서 검색해서 요약하고 오늘날씨에 맞는 외출복을 추천해줘'
prompt = dedent(f"""
사용자의 질문 의도를 분석하세요.

질문에 포함된 의도가 여러 개이면
반드시 모든 의도를 각각 분리하여 출력하세요.

예를 들어:
- 주식 + 뉴스 + 날씨
- 뉴스 + 검색
- 계산 + 주식

처럼 복합 질문이면
반드시 여러 개의 JSON 객체를 배열로 출력해야 합니다.

question_type 별로 tool_query를 생성하세요.
tool_query는 반드시 키워드중심으로 생성하세요 llm에 전달하는 문구가 아님을 명심해서 추론을하지말고 검색용 키워드로 매칭해주세요    
뉴스는 api검색할수 있는 키워드중심으로,
주식은 종목명만 추출하세요             
날씨도 검색용 키워드로 추출하세요                               

지원 가능한 question_type 예시:
- 날씨
- 뉴스
- 주식
- 검색
- 계산
- 지도

[중요 규칙]
- 질문에 포함된 모든 의도를 누락 없이 출력
- 반드시 JSON 배열(Array) 형식으로 출력
- 하나만 출력 금지
- 설명문 금지
- markdown 금지
- ```json 금지

[예시 입력]
어제 삼성전자 종가 알려주고 관련 뉴스와 오늘 날씨도 알려줘

[예시 출력]
[
    {{
        "question_type": "주식",
        "tool_query": "삼성전자",
        "reason": "삼성전자 종가 요청"
    }},
    {{
        "question_type": "뉴스",
        "tool_query": "삼성전자 관련 최근 뉴스 검색",
        "reason": "관련 뉴스 요청"
    }},
    {{
        "question_type": "날씨",
        "tool_query": "오늘 날씨 조회",
        "reason": "날씨 요청"
    }}
]

[질문]
{query}

[출력]
반드시 유효한 JSON 배열만 출력하세요.
""")

result = llm.generate(prompt)
json_result = json.loads(result)
json_result

[{'question_type': '주식',
  'tool_query': '삼성전자',
  'reason': '어제 삼성전자 주식 종가 조회 요청'},
 {'question_type': '뉴스',
  'tool_query': '삼성전자 종가와 비슷한 종목 관련 뉴스 요약',
  'reason': '해당 종가와 비슷한 종목을 뉴스에서 검색해 요약 요청'},
 {'question_type': '날씨',
  'tool_query': '오늘 날씨',
  'reason': '오늘 날씨에 맞는 외출복 추천 요청'}]

In [ ]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()    
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)
        print(result)

        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [44]:
search_naver_news('삼성전자 관련 최근 뉴스 검색')

200
{'lastBuildDate': 'Wed, 27 May 2026 12:42:49 +0900', 'total': 4098, 'start': 1, 'display': 3, 'items': [{'title': '‘N% 성과급’ 요구 산업계 전반 번져…재계 “투자 여력 위축” 우려', 'originallink': 'https://www.sedaily.com/article/20047127?ref=naver', 'link': 'https://n.news.naver.com/mnews/article/011/0004623517?sid=101', 'description': '독자 유형별 맞춤 <b>뉴스</b> 6개를 선별해 제공합니다. ■ 성과급 전쟁의 산업 전반 확산: <b>삼성전자</b>(005930) 노사... 정보 <b>검색</b>을 넘어 전문 실무를 직접 수행하는 단계로 빠르게 접어들었다. 현대차 에이미는 경쟁사 제품... ', 'pubDate': 'Fri, 22 May 2026 07:49:00 +0900'}, {'title': '[Who Is ?] 표경원 애경케미칼 대표이사 부사장', 'originallink': 'https://www.businesspost.co.kr/BP?command=article_view&num=437623', 'link': 'https://www.businesspost.co.kr/BP?command=article_view&num=437623', 'description': '애경케미칼은 <b>최근</b> 3년간 연구개발비를 꾸준히 늘렸다. 2023년 210억7400만 원, 2024년 210억9900만 원, 2025년... &lt;연합<b>뉴스</b>&gt; 1995년 <b>삼성</b>자동차에 입사했다. 1999년 <b>삼성전자</b>로 이동했다. 2002년 베인앤컴퍼니(Bain &amp; Company)... ', 'pubDate': 'Fri, 22 May 2026 07:02:00 +0900'}, {'title': '[K-VIBE] 전태

[{'title': '“최대 6억은 열어줬다”… 삼성전자 합의안 통과됐지만, DX는 사실상 등...',
  'content': '삼성전자 노사 임금·성과급 잠정합의안이 결국 조합원 투표를 통과했습니다. 총파업 가능성까지 거론됐던 노사 충돌은 일단 멈췄습니다. 하지만 투표 종료 직후 삼성전자 내부에서 더 강하게 번진 건 안도의... ',
  'date': '2026-05-27',
  'link': 'https://n.news.naver.com/mnews/article/661/0000077005?sid=101'},
 {'title': '삼성전자 노사, 2026년 임금협약 최종 서명… “의미 있는 합의 도달”',
  'content': '삼성전자와 노동조합 공동교섭단이 2026년 임금협약에 최종 서명했다. 삼성전자는 27일 경기 용인시 기흥에 있는 ‘더 유니버스’(The UniverSE)에서 노동조합 공동교섭단과 ‘2026년 임금협약 조인식’을 진행했다고... ',
  'date': '2026-05-27',
  'link': 'https://n.news.naver.com/mnews/article/366/0001167371?sid=105'},
 {'title': '반년간 이어진 줄다리기 마침표…삼성전자 노사 임금협약 체결',
  'content': '삼성전자 노사가 2026년 임금협약 체결을 공식적으로 매듭지었다. 노사는 향후 글로벌 경쟁력 강화를 위해 상호 협력하기로 뜻을 모았다. 27일 삼성전자에 따르면 회사는 이날 노동조합 공동교섭단(이하 공동교섭단)과... ',
  'date': '2026-05-27',
  'link': 'https://n.news.naver.com/mnews/article/005/0001851338?sid=101'},
 {'title': '‘N% 성과급’ 요구 산업계 전반 번져…재계 “투자 여력 위축” 우려',
  'content': '독자 유형별 맞춤 뉴스 6개를 선별해 제공합니다. ■ 성과급 전쟁의 산업 전반 확산: 삼성전자(005930